In [ ]:
#mount model

archive_path = "/content/drive/MyDrive/uni /tesi/tesi_colab.tar.gz"

!mkdir -p /content/tesi
!tar -xzf "{archive_path}" -C /content/tesi
print("✅ Estrazione completata con successo!")

✅ Estrazione completata con successo!


In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/uni /tesi/")
unpatched_infer_results_path = ROOT / "unpatched_infer_results.jsonl"
patched_infer_results_path = ROOT / "patched_infer_results.jsonl"
ground_truth_folder = "/content/drive/MyDrive/uni /tesi/HarmonixSet"

In [ ]:
!pip install mir_eval msaf loguru fire

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.8/531.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.2 MB/s eta 0:00:00


# we run eval

first on processed_results.jsonl

and then on processed_results_patched.jsonl - fp16_songs.txt

SongFormer outputs can be converted using the utility script src/SongFormer/utils/convert_res2msa_txt.py

In [ ]:
# expand jsonl in json folders
import json
def expand_jsonl(path: Path):
  filename = path.stem
  dir_path = Path(ROOT / filename)
  dir_path.mkdir(parents=True, exist_ok=True)
  with open(path, "r") as f:
    lines = f.readlines()
  for line in lines:
    line = json.loads(line)
    with open(dir_path / f"{line['song_id']}.json", "w") as f:
      json.dump(line["content"], f)

  return dir_path




In [ ]:
unpatched_infer_folder_path = expand_jsonl(unpatched_infer_results_path)
patched_infer_folder_path = expand_jsonl(patched_infer_results_path)

In [ ]:
# convert songformer results
def convert_songformer_results(path):
  filename = path.stem
  dir_path = ROOT / f"converted_{filename}"

  !python /content/tesi/SongFormer/src/SongFormer/utils/convert_res2msa_txt.py "$path" "$dir_path"

  return dir_path

In [ ]:
unpatched_converted_folder_path = convert_songformer_results(unpatched_infer_folder_path)
patched_converted_folder_path = convert_songformer_results(patched_infer_folder_path)

Input folder: /content/drive/MyDrive/uni /tesi/unpatched_infer_results
Output folder: /content/drive/MyDrive/uni /tesi/converted_unpatched_infer_results
--------------------------------------------------
Found 44 JSON files
✓ Processed: BHX_0799_lovesexmagic.json -> BHX_0799_lovesexmagic.txt
✓ Processed: BHX_0112_giveitup.json -> BHX_0112_giveitup.txt
✓ Processed: BHX_0197_nothinonyou.json -> BHX_0197_nothinonyou.txt
✓ Processed: BHX_0491_skyscraper.json -> BHX_0491_skyscraper.txt
✓ Processed: BHX_0450_more.json -> BHX_0450_more.txt
✓ Processed: BHX_0392_givemeyourhand.json -> BHX_0392_givemeyourhand.txt
✓ Processed: BHX_0770_kingofmycastle.json -> BHX_0770_kingofmycastle.txt
✓ Processed: BHX_0592_blameit.json -> BHX_0592_blameit.txt
✓ Processed: BHX_0807_makemebetter.json -> BHX_0807_makemebetter.txt
✓ Processed: BHX_0248_screamingfor.json -> BHX_0248_screamingfor.txt
✓ Processed: BHX_0668_fallinganthem.json -> BHX_0668_fallinganthem.txt
✓ Processed: BHX_0811_miami2ibiza.json -> BHX_0

In [ ]:
#patch eval.sh
%%writefile /content/tesi/SongFormer/src/SongFormer/eval.sh
#!/bin/bash
set -e

if [ "$#" -lt 3 ]; then
    echo "Error: Missing required arguments."
    echo "Usage: $0 <EST_DIR> <ANN_DIR> <OUTPUT_DIR>"
    exit 1
fi

EST_DIR="$1"
ANN_DIR="$2"
OUTPUT_DIR="$3"

# Resolve directory of this script (/content/tesi/SongFormer/src/SongFormer)
SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" &>/dev/null && pwd)"

export CUDA_VISIBLE_DEVICES=-1
export PYTHONPATH="$SCRIPT_DIR:$PYTHONPATH"

export HYDRA_FULL_ERROR=1
export OMP_NUM_THREADS=1
export MPI_NUM_THREADS=1
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1

echo "Evaluating: $EST_DIR against $ANN_DIR"
echo "Output directory: $OUTPUT_DIR"
mkdir -p "$OUTPUT_DIR"

python "$SCRIPT_DIR/evaluation/eval_infer_results.py" \
    --ann_dir "$ANN_DIR" \
    --est_dir "$EST_DIR" \
    --output_dir "$OUTPUT_DIR" \
    --prechorus2what verse

Overwriting /content/tesi/SongFormer/src/SongFormer/eval.sh


In [ ]:
def run_eval(EST_DIR, ANN_DIR, OUTPUT_DIR):
  # Run evaluation
  !bash /content/tesi/SongFormer/src/SongFormer/eval.sh "$EST_DIR" "$ANN_DIR" "$OUTPUT_DIR"

In [ ]:
# run eval for unpatched
run_eval(unpatched_converted_folder_path, ground_truth_folder, ROOT / "unpatched_eval_results")

Evaluating: /content/drive/MyDrive/uni /tesi/converted_unpatched_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/unpatched_eval_results
2026-09-20 18:08:33.779 | INFO     | __main__:main:96 - Common number of files: 44
  0% 0/44 [00:00<?, ?it/s]2026-09-20 18:08:33.784 | INFO     | __main__:main:103 - Processing BHX_0748_imyours.txt
  2% 1/44 [00:00<00:25,  1.67it/s]2026-09-20 18:08:34.384 | INFO     | __main__:main:103 - Processing BHX_0770_kingofmycastle.txt
  5% 2/44 [00:01<00:24,  1.68it/s]2026-09-20 18:08:34.975 | INFO     | __main__:main:103 - Processing BHX_0668_fallinganthem.txt
  7% 3/44 [00:01<00:24,  1.70it/s]2026-09-20 18:08:35.555 | INFO     | __main__:main:103 - Processing BHX_0763_iwasafool.txt
  9% 4/44 [00:02<00:22,  1.81it/s]2026-09-20 18:08:36.054 | INFO     | __main__:main:103 - Processing BHX_0022_betteroffalone.txt
 11% 5/44 [00:02<00:21,  1.85it/s]2026-09-20 18:08:36.572 | INFO     | __main__:ma

In [ ]:
# run eval for patched
run_eval(patched_converted_folder_path, ground_truth_folder, ROOT / "patched_eval_results")

Evaluating: /content/drive/MyDrive/uni /tesi/converted_patched_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/patched_eval_results
2026-09-20 18:23:10.594 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-20 18:23:10.595 | INFO     | __main__:main:103 - Processing BHX_0392_givemeyourhand.txt
2026-09-20 18:23:10.665 | INFO     | __main__:main:103 - Processing BHX_0866_pokerfacedave.txt
  4% 2/50 [00:00<00:05,  8.53it/s]2026-09-20 18:23:10.829 | INFO     | __main__:main:103 - Processing BHX_0653_dynamite.txt
  6% 3/50 [00:00<00:05,  8.68it/s]2026-09-20 18:23:10.942 | INFO     | __main__:main:103 - Processing BHX_0067_deep.txt
2026-09-20 18:23:11.004 | INFO     | __main__:main:103 - Processing BHX_0319_withoutyou.txt
 10% 5/50 [00:00<00:04, 11.18it/s]2026-09-20 18:23:11.077 | INFO     | __main__:main:103 - Processing BHX_0961_unfaithfulfemme.txt
2026-09-20 18:23:11.140 | INFO 

In [ ]:
# run eval for patched - fp16 songs
import os
# Read fp16 songs into a set for efficient lookup
fp16_songs = set()
fp16_songs_file = ROOT / "fp16_songs.txt"

if os.path.exists(fp16_songs_file):
    with open(fp16_songs_file, 'r') as f:
        for line in f:
            fp16_songs.add(line.strip())

print(fp16_songs)

{'BHX_0653_dynamite', 'BHX_0276_texasflood', 'BHX_0928_takeabow', 'BHX_0797_loveisgone', 'BHX_0866_pokerfacedave', 'BHX_0913_somuchlove'}


In [ ]:
import shutil

def remove_fp16_songs(source_folder: Path, fp16_songs: set):
  dest_folder = source_folder.parent / (source_folder.name + "_non_fp16")
  dest_folder.mkdir(parents=True, exist_ok=True)

  for file_path in source_folder.iterdir():
    if file_path.is_file() and file_path.suffix == '.txt': # Assuming converted files are .txt
      song_id = file_path.stem # Extracts 'BHX_0653_dynamite' from 'BHX_0653_dynamite.txt'
      if song_id not in fp16_songs:
        shutil.copy(file_path, dest_folder)
  print(f"Created filtered folder: {dest_folder} with {len(list(dest_folder.iterdir()))} files.")
  return dest_folder

patched_converted_folder_path = remove_fp16_songs(patched_converted_folder_path, fp16_songs)

Created filtered folder: /content/drive/MyDrive/uni /tesi/converted_patched_infer_results_non_fp16 with 44 files.


In [ ]:
# run eval for patched_non_fp16

run_eval(patched_converted_folder_path, ground_truth_folder, ROOT / "patched_non_fp16_eval_results")

Evaluating: /content/drive/MyDrive/uni /tesi/converted_patched_infer_results_non_fp16 against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/patched_non_fp16_eval_results
2026-09-20 18:14:51.759 | INFO     | __main__:main:96 - Common number of files: 44
  0% 0/44 [00:00<?, ?it/s]2026-09-20 18:14:51.761 | INFO     | __main__:main:103 - Processing BHX_0851_openurmind.txt
2026-09-20 18:14:51.801 | INFO     | __main__:main:103 - Processing BHX_0123_heavyduty.txt
  5% 2/44 [00:00<00:02, 18.98it/s]2026-09-20 18:14:51.866 | INFO     | __main__:main:103 - Processing BHX_0576_barbrastreisand.txt
2026-09-20 18:14:51.908 | INFO     | __main__:main:103 - Processing BHX_0248_screamingfor.txt
2026-09-20 18:14:51.957 | INFO     | __main__:main:103 - Processing BHX_0332_alejandro.txt
 11% 5/44 [00:00<00:01, 20.18it/s]2026-09-20 18:14:52.010 | INFO     | __main__:main:103 - Processing BHX_0676_figure8.txt
2026-09-20 18:14:52.055 | INFO     | __main__:mai

In [ ]:
# now we show that the patch didn't affect the model so we prove that unpatched and patched_non_fp16 are the same

# TODO

In [ ]:

smsa_infer_results_path = ROOT / "SMSA_infer_results"

run_eval(smsa_infer_results_path, ground_truth_folder, ROOT / "smsa_eval_results")


Evaluating: /content/drive/MyDrive/uni /tesi/SMSA_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/smsa_eval_results
2026-09-23 21:25:03.380 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-23 21:25:03.381 | INFO     | __main__:main:103 - Processing BHX_0578_bealright.txt
  2% 1/50 [00:00<00:47,  1.03it/s]2026-09-23 21:25:04.354 | INFO     | __main__:main:103 - Processing BHX_0555_allaroundme.txt
  4% 2/50 [00:01<00:46,  1.03it/s]2026-09-23 21:25:05.317 | INFO     | __main__:main:103 - Processing BHX_0797_loveisgone.txt
  6% 3/50 [00:02<00:46,  1.02it/s]2026-09-23 21:25:06.311 | INFO     | __main__:main:103 - Processing BHX_0913_somuchlove.txt
  8% 4/50 [00:03<00:44,  1.03it/s]2026-09-23 21:25:07.276 | INFO     | __main__:main:103 - Processing BHX_0668_fallinganthem.txt
 10% 5/50 [00:04<00:42,  1.05it/s]2026-09-23 21:25:08.182 | INFO     | __main__:main:103 - Processing BHX_

In [ ]:
smsa_flash_infer_results_path = ROOT / "smsa_flash_infer_results"

run_eval(smsa_flash_infer_results_path, ground_truth_folder, ROOT / "smsa_flash_eval_results")


Evaluating: /content/drive/MyDrive/uni /tesi/smsa_flash_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/smsa_flash_eval_results
2026-09-24 13:03:46.694 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-24 13:03:46.695 | INFO     | __main__:main:103 - Processing BHX_0676_figure8.txt
  2% 1/50 [00:00<00:34,  1.41it/s]2026-09-24 13:03:47.403 | INFO     | __main__:main:103 - Processing BHX_0112_giveitup.txt
  4% 2/50 [00:01<00:33,  1.42it/s]2026-09-24 13:03:48.106 | INFO     | __main__:main:103 - Processing BHX_0799_lovesexmagic.txt
  6% 3/50 [00:02<00:32,  1.44it/s]2026-09-24 13:03:48.791 | INFO     | __main__:main:103 - Processing BHX_0503_stucklikeglue.txt
  8% 4/50 [00:02<00:32,  1.42it/s]2026-09-24 13:03:49.502 | INFO     | __main__:main:103 - Processing BHX_0332_alejandro.txt
 10% 5/50 [00:03<00:31,  1.44it/s]2026-09-24 13:03:50.177 | INFO     | __main__:main:103 - Process

In [ ]:
smsa_flash_infer_results_path = ROOT / "smsa_lite_infer_results"

run_eval(smsa_flash_infer_results_path, ground_truth_folder, ROOT / "smsa_lite_eval_results")

Evaluating: /content/drive/MyDrive/uni /tesi/smsa_lite_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/smsa_lite_eval_results
2026-09-23 22:51:10.635 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-23 22:51:10.636 | INFO     | __main__:main:103 - Processing BHX_0484_rockthatbody.txt
  2% 1/50 [00:00<00:24,  1.96it/s]2026-09-23 22:51:11.146 | INFO     | __main__:main:103 - Processing BHX_0913_somuchlove.txt
  4% 2/50 [00:01<00:25,  1.85it/s]2026-09-23 22:51:11.708 | INFO     | __main__:main:103 - Processing BHX_0378_effinperfect.txt
  6% 3/50 [00:01<00:23,  1.96it/s]2026-09-23 22:51:12.180 | INFO     | __main__:main:103 - Processing BHX_0928_takeabow.txt
  8% 4/50 [00:02<00:24,  1.91it/s]2026-09-23 22:51:12.725 | INFO     | __main__:main:103 - Processing BHX_0799_lovesexmagic.txt
 10% 5/50 [00:02<00:22,  1.99it/s]2026-09-23 22:51:13.193 | INFO     | __main__:main:103 - Proc

In [ ]:
direct_audio_infer_results_path = ROOT / "direct_audio_infer_results"

run_eval(direct_audio_infer_results_path, ground_truth_folder, ROOT / "direct_audio_eval_results")

Evaluating: /content/drive/MyDrive/uni /tesi/direct_audio_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/direct_audio_eval_results
2026-09-27 20:34:24.515 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-27 20:34:24.517 | INFO     | __main__:main:103 - Processing BHX_0780_letmebereal.txt
  2% 1/50 [00:00<00:35,  1.40it/s]2026-09-27 20:34:25.232 | INFO     | __main__:main:103 - Processing BHX_0273_technologic.txt
  4% 2/50 [00:01<00:33,  1.41it/s]2026-09-27 20:34:25.934 | INFO     | __main__:main:103 - Processing BHX_0913_somuchlove.txt
  6% 3/50 [00:02<00:33,  1.39it/s]2026-09-27 20:34:26.668 | INFO     | __main__:main:103 - Processing BHX_0653_dynamite.txt
  8% 4/50 [00:02<00:34,  1.32it/s]2026-09-27 20:34:27.486 | INFO     | __main__:main:103 - Processing BHX_0197_nothinonyou.txt
 10% 5/50 [00:03<00:32,  1.39it/s]2026-09-27 20:34:28.137 | INFO     | __main__:main:103 - P

In [ ]:

futga_infer_results_path = ROOT / "futga_infer_results"

run_eval(futga_infer_results_path, ground_truth_folder, ROOT / "futga_eval_results")


Evaluating: /content/drive/MyDrive/uni /tesi/futga_infer_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/drive/MyDrive/uni /tesi/futga_eval_results
2026-09-23 15:57:14.013 | INFO     | __main__:main:96 - Common number of files: 50
  0% 0/50 [00:00<?, ?it/s]2026-09-23 15:57:14.014 | INFO     | __main__:main:103 - Processing BHX_0666_fallin.txt
2026-09-23 15:57:14.050 | INFO     | __main__:main:103 - Processing BHX_0273_technologic.txt
2026-09-23 15:57:14.074 | INFO     | __main__:main:103 - Processing BHX_0780_letmebereal.txt
2026-09-23 15:57:14.104 | INFO     | __main__:main:103 - Processing BHX_0005_again.txt
  8% 4/50 [00:00<00:01, 33.07it/s]2026-09-23 15:57:14.136 | INFO     | __main__:main:103 - Processing BHX_0022_betteroffalone.txt
2026-09-23 15:57:14.163 | INFO     | __main__:main:103 - Processing BHX_0748_imyours.txt
2026-09-23 15:57:14.211 | INFO     | __main__:main:103 - Processing BHX_0503_stucklikeglue.txt
2026-09-23 15:57:14.258 | IN